# Matplotlib Subplots

> 📘 **Python Mastery** · Module 12 — Matplotlib · Lesson 4/5

Real analysis rarely needs one chart — it needs four, side by side, sharing one
story. This lesson teaches both ways to arrange multiple plots in a single
figure, and ends by building a complete student-analytics dashboard.

## 🎯 Learning Objectives

By the end of this lesson you will be able to:

- **Arrange** plots with the state-machine call `plt.subplot(rows, cols, n)`
- **Use** the object-oriented way: `fig, axes = plt.subplots(...)` — and explain why it scales better
- **Share** x- or y-axes across related plots with `sharex=` / `sharey=`
- **Fix** overlapping text with `tight_layout()`
- **Loop** over a grid of Axes using `axes.ravel()`
- **Assemble** a 2×2 dashboard with `suptitle` and custom row heights (`height_ratios`)

## 1. Why Multiple Plots in One Figure?

Two reasons professionals always reach for subplots:

1. **Fair comparison** — separate figures get separate axis scales; your eye
   can't honestly compare bar heights across two different windows.
2. **Dashboards** — trend, breakdown, distribution, and relationship belong
   together on ONE canvas for ONE dataset.

A figure is the *frame*; subplots are several *photos* hung in it.

**Syntax:** the two vocabularies you are about to learn.

```python
plt.subplot(rows, cols, n)        # state-machine style: activate slot n...
plt.plot(...)                     # ...then draw into it

fig, axes = plt.subplots(2, 2)    # object-oriented style: make them all,
axes[0, 0].plot(...)              # keep handles, address each directly
```

**Example:** first, feel the problem — these two charts were shown separately,
so comparing their peaks means remembering pixel heights.

In [ ]:
import matplotlib.pyplot as plt

week = ["Sat", "Sun", "Mon", "Tue", "Wed"]
study = [2, 3, 5, 4, 6]
sleep = [7, 8, 6, 9, 8]

plt.plot(week, study, marker="o")
plt.title("Study - separate figure #1")
plt.show()

plt.plot(week, sleep, marker="o")
plt.title("Sleep - separate figure #2")
plt.show()

# Which fluctuates more? Impossible to judge honestly - different scales,
# different positions. Subplots fix this.

## 2. The State-Machine Way — `plt.subplot(rows, cols, n)`

`plt.subplot(2, 3, 4)` splits the figure into a **2-row × 3-column** grid and
makes slot **4** the *current* Axes. Every `plt.*` call afterwards lands there,
until you activate another slot.

⚠️ Counting starts at **1**, not 0 — slots run left-to-right, top-to-bottom:
`1 2 3` on the top row, `4 5 6` below. (Legacy shorthand `plt.subplot(234)`
means exactly the same thing.)

**Syntax:**
```python
plt.subplot(1, 2, 1)      # 1 row, 2 columns, LEFT slot
plt.plot(...)
plt.title("Left")
plt.subplot(1, 2, 2)      # same grid, RIGHT slot
plt.plot(...)
plt.title("Right")
```

**Example:** study vs sleep side by side — now comparison is honest.

In [ ]:
import matplotlib.pyplot as plt

week = ["Sat", "Sun", "Mon", "Tue", "Wed"]
study = [2, 3, 5, 4, 6]
sleep = [7, 8, 6, 9, 8]

plt.figure(figsize=(10, 4))       # wide canvas for two panels

plt.subplot(1, 2, 1)              # slot 1 (left)
plt.plot(week, study, marker="o")
plt.title("Study hours")
plt.xlabel("Day")

plt.subplot(1, 2, 2)              # slot 2 (right)
plt.plot(week, sleep, marker="o", color="#2a9d8f")
plt.title("Sleep hours")
plt.xlabel("Day")

plt.show()
# Same y-range would make it even fairer -> that's sharey, coming up.

## 3. The Object-Oriented Way — `fig, axes = plt.subplots(...)`

One call creates the Figure AND every Axes, handing you Python objects to keep:

```python
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
```

`axes` is a NumPy array of Axes objects — for a 2×2 grid, address them like a
matrix: `axes[0, 0]` top-left, `axes[1, 1]` bottom-right. Methods change name
slightly: not `plt.title(t)` but `ax.set_title(t)`.

| pyplot (state machine) | Object-oriented |
|---|---|
| `plt.title("t")` | `ax.set_title("t")` |
| `plt.xlabel("x")` | `ax.set_xlabel("x")` |
| `plt.xlim(0, 10)` | `ax.set_xlim(0, 10)` |
| `plt.grid(True)` | `ax.grid(True)` |
| `plt.legend()` | `ax.legend()` |

> 🔍 **Under the Hood:** the `plt.*` functions work through module-level global
> state — a stack of "current figure / current axes" managed internally by
> pyplot's figure manager. `plt.subplot(2, 2, 3)` literally *mutates that global
> pointer*, so any re-ordering of your lines silently sends data to the wrong
> panel. The OO interface bypasses the globals entirely: you hold explicit
> references and pass `ax` wherever it's needed (functions, loops, classes).
> That is why anything beyond a quick sketch — dashboards, loops, reusable plot
> helpers, GUIs — is written object-oriented.

✅ **Recommended convention:** from today, prefer
`fig, ax = plt.subplots(figsize=(8, 5))` **even for a single plot**, then use
`ax.plot`, `ax.set_title`, `ax.set_xlabel`. It costs nothing now and scales
forever.

**Why OO scales better:**
- **Explicit handles** — no "whatever is current" surprises when code is reordered
- **Loopable** — `for ax in axes.ravel():` styles 20 panels in 3 lines
- **Passable** — a helper function `def style(ax): ...` works on any panel
- **Composable** — insets, twin axes, and complex layouts demand object handles

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 10, 100)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))   # 2x2 grid, keep the handles

axes[0, 0].plot(x, np.sin(x), color="#2a6f97")          # top-left
axes[0, 0].set_title("sin(x)")

axes[0, 1].plot(x, np.cos(x), color="#e76f51")          # top-right
axes[0, 1].set_title("cos(x)")

axes[1, 0].plot(x, x ** 2, color="#2a9d8f")             # bottom-left
axes[1, 0].set_title("x squared")

axes[1, 1].plot(x, np.exp(-x / 5), color="#6a4c93")     # bottom-right
axes[1, 1].set_title("decaying exponential")

for ax in axes.ravel():               # one loop styles EVERY panel
    ax.set_xlabel("x")
    ax.grid(True, linestyle="--", alpha=0.5)

plt.show()

## 4. Sharing Axes — `sharex=` and `sharey=`

Related panels usually deserve identical scales. Without sharing, each Axes
auto-scales independently and silently lies about relative size.

- `sharey=True` — equal y-limits everywhere (compare magnitudes!)
- `sharex=True` — equal x-limits (aligned timelines)
- Tick labels de-clutter automatically on inner panels.

**Syntax:**
```python
fig, axes = plt.subplots(2, 1, sharex=True)   # stacked, aligned timeline
```

**Example:** two riders' weekly earnings — shared y makes the gap undeniable.

In [ ]:
import matplotlib.pyplot as plt

days   = ["Sat", "Sun", "Mon", "Tue", "Wed", "Thu"]
karim  = [35, 42, 38, 51, 47, 55]
priya  = [40, 38, 48, 55, 50, 62]

fig, axes = plt.subplots(2, 1, sharex=True, sharey=True, figsize=(8, 6))

axes[0].plot(days, karim, marker="o", color="#2a6f97")
axes[0].set_title("Karim's earnings")

axes[1].plot(days, priya, marker="s", color="#e76f51")
axes[1].set_title("Priya's earnings")

for ax in axes:
    ax.set_ylabel("Earnings")
    ax.grid(True, linestyle="--", alpha=0.5)

plt.show()
# Identical y-scale: heights mean the SAME thing in both panels.

## 5. `tight_layout()` — Fixing the Squish

With several panels, titles and tick labels collide. `fig.tight_layout()`
measures everything and nudges panels apart automatically. (Modern alternative:
pass `layout="constrained"` or `constrained_layout=True` when creating the grid.)

**Syntax:**
```python
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
...                          # fill panels
fig.tight_layout()           # last line before show()
```

**Example:** a deliberately crowded 2×2 grid, rescued by one line.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.random.seed(3)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

data_sets = {
    "Math":     np.clip(np.random.normal(75, 9, 60), 30, 100),
    "Physics":  np.clip(np.random.normal(66, 13, 60), 30, 100),
    "Chemistry": np.clip(np.random.normal(71, 11, 60), 30, 100),
    "Biology":  np.clip(np.random.normal(78, 8, 60), 30, 100),
}

positions = [(0, 0), (0, 1), (1, 0), (1, 1)]

for (subject, scores), pos in zip(data_sets.items(), positions):
    axes[pos].hist(scores, bins=10, edgecolor="white", color="#457b9d")
    axes[pos].set_title(f"{subject} score distribution - long title!")
    axes[pos].set_xlabel("Score")
    axes[pos].set_ylabel("Students")

fig.tight_layout()      # <- without this, titles overlap the row above
plt.show()

## 6. `axes.ravel()` — Looping Over Any Grid

A 2×2 `axes` array is 2-D, but `.ravel()` flattens it to a plain sequence —
so one `zip()` fills any grid, no matter its shape. This pattern appears in
virtually every professional plotting script.

**Syntax:**
```python
for ax, item in zip(axes.ravel(), my_data):
    ax.plot(item)
```

**Example:** five subjects, one loop, zero copy-paste.

In [ ]:
import matplotlib.pyplot as plt

subjects = ["Math", "Physics", "Chemistry", "Biology", "English"]
marks    = [88, 74, 91, 79, 84]

fig, axes = plt.subplots(1, 5, figsize=(14, 3), sharey=True)

for ax, subj, mk in zip(axes.ravel(), subjects, marks):
    ax.bar(subj, mk, color="#2a6f97" if mk >= 80 else "#adb5bd")
    ax.set_title(subj, fontsize=10)
    ax.set_ylim(0, 100)

axes[0].set_ylabel("Marks")
plt.show()

# Change the grid to (5, 1) or (3, 2)... the SAME loop still works.

## 7. Project — a 2×2 Student Analytics Dashboard

Everything combines here: one dataset, four questions, one figure.

| Panel | Chart | Question |
|---|---|---|
| `[0, 0]` | Line | How did weekly study hours evolve? |
| `[0, 1]` | Bar | Which subject scored highest? |
| `[1, 0]` | Histogram | How are all marks distributed? |
| `[1, 1]` | Scatter | Do more hours really mean higher marks? |

Finish with `fig.suptitle(...)` for the overall headline (distinct from each
panel's `ax.set_title`), plus `tight_layout` — giving suptitle headroom via
its `y`/rect tweak if needed.

**Syntax:**
```python
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].plot(...); axes[0, 1].bar(...)
axes[1, 0].hist(...); axes[1, 1].scatter(...)
fig.suptitle("Overall headline", fontsize=16, fontweight="bold")
fig.tight_layout()
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
# ---- one dataset: 15 students tracked over time -----------------------
weeks       = list(range(1, 9))
study_avg   = [3.0, 3.5, 4.0, 3.5, 5.0, 5.5, 6.0, 6.5]          # class average h/week
subjects    = ["Math", "Physics", "Chemistry", "Biology", "English"]
subj_marks  = [88, 74, 91, 79, 84]                              # avg mark per subject
all_marks   = np.clip(np.random.normal(76, 10, 120), 40, 100)   # every exam taken
stu_hours   = np.array([2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 8, 9, 5, 4, 7])
stu_marks   = 45 + stu_hours * 5.2                              # strongly related

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# [0,0] line - trend
axes[0, 0].plot(weeks, study_avg, marker="o", color="#2a6f97")
axes[0, 0].set_title("Weekly Study Hours (class average)")
axes[0, 0].set_xlabel("Week")
axes[0, 0].set_ylabel("Hours / week")

# [0,1] bar - comparison
bars = axes[0, 1].bar(subjects, subj_marks,
                      color=["#adb5bd", "#adb5bd", "#e76f51", "#adb5bd", "#adb5bd"])
axes[0, 1].set_title("Average Mark per Subject")
axes[0, 1].set_ylim(0, 100)
axes[0, 1].tick_params(axis="x", labelrotation=30)

# [1,0] histogram - distribution
axes[1, 0].hist(all_marks, bins=12, edgecolor="white", color="#457b9d")
axes[1, 0].set_title("Distribution of All Exam Marks")
axes[1, 0].set_xlabel("Mark")
axes[1, 0].set_ylabel("Number of exams")

# [1,1] scatter - relationship
axes[1, 1].scatter(stu_hours, stu_marks, s=60, alpha=0.8,
                   color="#2a9d8f", edgecolor="black")
axes[1, 1].set_title("Study Hours vs Marks (per student)")
axes[1, 1].set_xlabel("Hours studied / week")
axes[1, 1].set_ylabel("Exam mark")

fig.suptitle("Student Analytics Dashboard", fontsize=16, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])   # leave headroom for the suptitle

plt.show()

## 8. Custom Row Heights — `gridspec_kw={"height_ratios": ...}`

Not every panel deserves equal space. Pass `gridspec_kw` (keyword arguments
forwarded to the underlying GridSpec layout engine) to control proportions:
`[2, 1]` makes row 1 twice as tall as row 2. Perfect for "big main chart,
small companion chart".

**Syntax:**
```python
fig, axes = plt.subplots(
    2, 1,
    gridspec_kw={"height_ratios": [2, 1]},   # top panel twice as tall
    sharex=True,
)
```

**Example:** revenue takes the stage; profit rides along underneath.

In [ ]:
import matplotlib.pyplot as plt

months  = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
revenue = [420, 380, 510, 480, 590, 650]
profit  = [42, 30, 65, 52, 80, 95]

fig, axes = plt.subplots(
    2, 1,
    figsize=(10, 6),
    sharex=True,
    gridspec_kw={"height_ratios": [2, 1]},   # revenue gets 2/3 of the height
)

axes[0].plot(months, revenue, marker="o", linewidth=2.5, color="#2a6f97")
axes[0].set_title("Revenue (main story)")
axes[0].set_ylabel("Thousand BDT")

axes[1].bar(months, profit, color="#e9c46a")
axes[1].set_title("Profit (companion)")
axes[1].set_ylabel("Thousand BDT")

fig.tight_layout()
plt.show()

## 9. Bonus — Label-Based Grids with `subplot_mosaic`

Counting slots gets fiddly for irregular layouts. `plt.subplot_mosaic` lets you
*draw the layout with words*: repeat a label across cells to make that panel
span them, then use the label as a key.

**Syntax:**
```python
fig, axd = plt.subplot_mosaic([
    ["main",  "side"],
    ["main",  "bottom"],      # "main" spans both rows of the left column
])
axd["main"].plot(...)         # address panels by NAME, not by number
```

**Example:** one tall trend chart plus two side panels — no slot arithmetic.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(11)
months  = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
revenue = [420, 380, 510, 480, 590, 650]
profit  = [42, 30, 65, 52, 80, 95]

fig, axd = plt.subplot_mosaic(
    [["main", "side"],
     ["main", "bottom"]],
    figsize=(11, 6),
)

axd["main"].plot(months, revenue, marker="o", color="#2a6f97", linewidth=2.5)
axd["main"].set_title("Revenue - spans both rows")

axd["side"].bar(months, profit, color="#e9c46a")
axd["side"].set_title("Profit")
axd["side"].tick_params(axis="x", labelrotation=90, labelsize=8)

axd["bottom"].hist(np.clip(np.random.normal(500, 90, 200), 250, 900),
                   bins=15, edgecolor="white", color="#457b9d")
axd["bottom"].set_title("Daily revenue spread")

fig.tight_layout()
plt.show()

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Treating `subplot` slots as 0-based | Off-by-one: wrong panel or `ValueError` | Slots count from **1**: `(2,3,n)` allows n = 1…6 |
| Indexing a 2×2 grid with `axes[1]` | `IndexError: too many indices` — it's 2-D | Use `axes[row, col]`, or flatten with `axes.ravel()` |
| Using `plt.title(...)` after creating many panels | Titles only the LAST activated Axes | Call `ax.set_title(...)` on each Axes |
| Skipping `tight_layout()` on dense grids | Titles/labels overlap into garbage | Add `fig.tight_layout()` before `show()` |
| Forgetting `figsize` | Panels render thumbnail-sized | `plt.subplots(..., figsize=(12, 8))` |
| Comparing panels with independent auto-scales | Heights imply differences that don't exist | `sharex=True` / `sharey=True` |

## 💡 Best Practices & Pro Tips

- Default to the OO interface: `fig, ax = plt.subplots()`. Reach back for
  `plt.subplot(r, c, n)` only in throwaway exploratory cells.
- Keep one consistent y-axis meaning per dashboard row; readers scan rows, not panels.
- `suptitle` is the headline, `set_title` is the caption — never repeat the same words in both.
- Reuse a palette dict `{series_name: color}` so every panel colors entities identically.
- 🤖 **AI-engineering relevance:** the classic ML training dashboard is EXACTLY
  this lesson — a 2×2 grid of train-loss, validation-loss, train-accuracy,
  validation-accuracy built with `subplots` + `ravel()` + `suptitle`.
  Faceted "small multiples" of model comparisons use the identical machinery.

## 📌 Summary

| Tool | What it does | Example |
|---|---|---|
| `plt.subplot(r, c, n)` | Activate grid slot n (1-based!) | `plt.subplot(1, 2, 2)` |
| `fig, axes = plt.subplots(2, 2)` | Create grid, keep object handles | `figsize=(12, 8)` |
| `axes[i, j]` | Address a panel like a matrix | `axes[0, 1].plot(...)` |
| `axes.ravel()` | Flatten any grid for looping | `for ax in axes.ravel():` |
| `sharex=True` / `sharey=True` | Align scales across panels | honest comparisons |
| `fig.tight_layout()` | Auto-fix overlapping furniture | call before `show()` |
| `fig.suptitle(t)` | Figure-level headline | `fontsize=16, fontweight="bold"` |
| `gridspec_kw={"height_ratios": [2, 1]}` | Custom row/column proportions | big-main + small-companion |
| `plt.subplot_mosaic([["a", "b"], ["a", "c"]])` | Label-based layouts; panels span cells by name | `axd["a"].plot(...)` |

**Key takeaways**

- `plt.subplot` switches a global "current" panel; OO `subplots` hands you explicit objects — OO wins for anything real.
- Panel methods are `set_*`: `ax.set_title`, `ax.set_xlabel`, `ax.set_xlim`.
- Shared axes make multi-panel comparisons honest; `tight_layout` keeps them readable.
- Dashboard recipe: one dataset → 2×2 → line/bar/hist/scatter → `suptitle` + `tight_layout`.

## 🔗 Next Lesson

➡️ Continue to **[05_Customization](../05_Customization/notes.ipynb)** — zooming, annotations, threshold lines, clean spines, log scales, heatmaps, and a publication-quality finale.